# Day 4: RAG Fundamentals and Vector Databases

Day 3 was about fine-tuning, changing a model's weights. Today's about the other real way to fix a model that only knows what it was trained on: RAG, retrieval-augmented generation, where the model stays frozen and just gets handed real documents at question time instead.

Our headlines are single sentences, too short to meaningfully chunk, so this notebook fetches real full article bodies for a batch of our fool.com URLs first, then does the actual pipeline: chunk the text, embed the chunks, store them in a vector database, and run real semantic search against it.

In [1]:
import pandas as pd

## same news dataset used throughout the project, 843 rows as of the Day 3 growth run
df = pd.read_csv("news_dataset.csv")
print("Rows loaded:", len(df))
print(df.head())

## headlines are single sentences, not full articles -- decided to fetch real article
## body text for a batch of fool.com URLs (103 available), gives genuine multi-paragraph
## documents actually worth chunking, instead of chunking single-sentence headlines

from playwright.sync_api import sync_playwright
from concurrent.futures import ThreadPoolExecutor

## notebooks run their own asyncio event loop, and Playwright's sync API refuses to run
## inside any already-running loop -- running it in a separate thread sidesteps that entirely,
## since a fresh thread has no event loop of its own
fool_df = df[df["URL"].str.contains("fool.com", na=False)].head(30)
print(f"\nFetching {len(fool_df)} fool.com articles...")

def scrape_fool_articles(urls_df):
    documents = []
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page()
        for _, row in urls_df.iterrows():
            try:
                page.goto(row["URL"], wait_until="domcontentloaded", timeout=30000)
                page.wait_for_timeout(1000)
                body_text = page.locator(".article-body").first.inner_text()
                documents.append({
                    "url": row["URL"],
                    "title": row["Title"],
                    "category": row["Category"],
                    "text": body_text,
                })
            except Exception as e:
                print(f"Skipped {row['URL']}: {e}")
        browser.close()
    return documents

with ThreadPoolExecutor(max_workers=1) as executor:
    documents = executor.submit(scrape_fool_articles, fool_df).result()

print(f"\nFetched {len(documents)} full article documents")
print("First document length:", len(documents[0]["text"]), "characters")
print("First document preview:", documents[0]["text"][:200])

Rows loaded: 843
                                                 URL  \
0  https://www.fool.com/investing/2026/08/18/elon...   
1  https://www.fool.com/investing/2026/08/18/lisa...   
2  https://www.fool.com/investing/2026/08/18/wher...   
3  https://www.fool.com/coverage/stock-market-tod...   
4  https://www.fool.com/investing/2026/08/18/befo...   

                                               Title    Category  
0  Elon Musk Admits There Is Likely to Be Short-T...    Business  
1  Lisa Su Says AMD's Server Revenue Will Grow Mo...  Technology  
2  Where Will the Vanguard S&P 500 ETF Be in 20 Y...     Markets  
3  Stock Market Today, Aug. 18: CoreWeave Falls a...     Markets  
4  Before You Buy an AI Stock, Consider The Battl...  Technology  

Fetching 30 fool.com articles...



Fetched 30 full article documents
First document length: 3053 characters
First document preview: When Space Exploration Technologies Corp (
SPCX
+1.26%
), more commonly known as SpaceX, went public in June, much of the excitement centered on its future growth opportunities. Growth investors were 


All 30 fetched successfully, real multi-paragraph articles, not headlines. Each one's stored with its URL, title, and category as metadata, so anything retrieved later can get traced back to its actual source article.

In [2]:
##all rows were loaded succesfully, moving on to the chunking step
##chunking code w/ overlap
##this splits each document into smaller pieces that the model can actually work with
##overlap b/w chunks so info near a chunk boundary does not get cut off and lost
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap # move forward by (chunk_size - overlap), not chunk_size
    return chunks
all_chunks = []
for doc in documents:
    doc_chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "text": chunk,
            "url": doc["url"],
            "title": doc["title"],
            "category": doc["category"],
            "chunk_index": i,
        })
print(f"\n{len(documents)} documents split into {len(all_chunks)} chunks")
print("Average chunks per document:", len(all_chunks) / len(documents))
print("\nFirst document's chunks:")
first_doc_chunks = [c for c in all_chunks if c["url"] == documents[0]["url"]]
for c in first_doc_chunks:
    print(f"  chunk {c['chunk_index']} ({len(c['text'])} chars): {c['text'][:80]}...")


30 documents split into 282 chunks
Average chunks per document: 9.4

First document's chunks:
  chunk 0 (500 chars): When Space Exploration Technologies Corp (
SPCX
+1.26%
), more commonly known as...
  chunk 1 (500 chars): rm, and it's those growth opportunities that may lead to significant payoffs for...
  chunk 2 (500 chars):  for short-term pain.

Image source: Getty Images.

Musk admits the company is l...
  chunk 3 (500 chars): n the moon and Mars, predicting that he'll be criticized for falling short of ea...
  chunk 4 (500 chars): proposition to invest in the business because while the goals may take many year...
  chunk 5 (500 chars): Current Price
$154.64
KEY DATA POINTS
Market Cap
$2.1T
Market cap calculated usi...
  chunk 6 (500 chars): aceX went public, and already its share price has gone as high as $225 and as lo...
  chunk 7 (253 chars): Given how lofty the company's goals are, the safest option for investors is to t...


30 documents turned into 282 chunks, about 9-10 chunks per article. The overlap is doing its job, chunk boundaries don't land on clean 500-character marks since each new chunk starts 400 characters after the last one, not 500.

One real imperfection worth keeping: chunk 5 of the first document pulled in stock-widget boilerplate, "Current Price $154.42 KEY DATA POINTS Market Cap $2.1T," mixed in with real article text. Financial news sites often embed a live ticker widget right inside the article body div, and a plain text extraction has no way to tell that apart from the actual writing. Not something to fix here, just a real limitation of scraping article text this simply.

In [3]:
## ---------- embeddings ----------
## same core idea as GloVe embeddings, but this time using a model
## actually built for sentence/paragraph-level semantic search, not word-averaging
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in all_chunks]
embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)

print(f"\nEmbedded {len(embeddings)} chunks")
print("Embedding dimension:", len(embeddings[0]))

## ---------- vector database (ChromaDB) ----------
## ChromaDB stores the embeddings AND their metadata together, runs fully local,
## no separate server needed -- good fit for a single-user local project like this,
## unlike FAISS (a pure vector index, no built-in metadata storage) or Qdrant
## (a full production vector database server, more than this needs)
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="news_articles")

collection.add(
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"url": c["url"], "title": c["title"], "category": c["category"]} for c in all_chunks],
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
)

print(f"\nStored {collection.count()} chunks in the vector database")

## ---------- real vector search ----------
query = "What is Elon Musk saying about SpaceX?"
query_embedding = embed_model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
)

print(f"\n--- Search: {query} ---")
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"\n[{meta['category']}] {meta['title']} (distance: {dist:.3f})")
    print(doc[:150])
query2 = "How is Nvidia performing in the AI chip market?"
query2_embedding = embed_model.encode([query2])

results2 = collection.query(
    query_embeddings=query2_embedding.tolist(),
    n_results=3,
)

print(f"\n--- Search: {query2} ---")
for doc, meta, dist in zip(results2["documents"][0], results2["metadatas"][0], results2["distances"][0]):
    print(f"\n[{meta['category']}] {meta['title']} (distance: {dist:.3f})")
    print(doc[:150])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17615.94it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:  11%|█         | 1/9 [00:00<00:03,  2.20it/s]

...

Batches:  78%|███████▊  | 7/9 [00:00<00:00,  9.67it/s]

Batches: 100%|██████████| 9/9 [00:01<00:00, 11.82it/s]

Batches: 100%|██████████| 9/9 [00:01<00:00,  8.84it/s]


Embedded 282 chunks
Embedding dimension: 384



Stored 282 chunks in the vector database

--- Search: What is Elon Musk saying about SpaceX? ---

[Business] Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors (distance: 0.654)
rm, and it's those growth opportunities that may lead to significant payoffs for investors in the end. CEO Elon Musk's grand visions helped make Tesla

[Business] Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors (distance: 0.742)
 for short-term pain.

Image source: Getty Images.

Musk admits the company is likely to fall short of quarterly earnings estimates

In a recent inter

[Business] Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors (distance: 1.026)
When Space Exploration Technologies Corp (
SPCX
+1.12%
), more commonly known as SpaceX, went public in June, much of the excitement centered on its f

--- Search: How is Nvidia performing in the AI chip market? ---

[Business] AMD vs. Nvidia: 1 Metric Tells Me Which Is Clearly the

Both queries pulled the correct source article every time, ranked by distance, lower means more similar. Neither query used the article's exact wording, the Nvidia query never says AMD or GPU but still matched an AMD-vs-Nvidia GPU article.

Only 30 articles in the batch, so all three results for each query came from the same source article.

RAG means Retrieval-Augmented Generation, the model stays frozen and answers using real documents handed to it at question time instead of relying only on what it learned during training. The retrieval part is exactly what the vector search above is doing, turning a question into an embedding and pulling back the closest matching chunks.

Fine-tuning and RAG solve the same underlying problem, a model only knowing what it was trained on, in two different ways. Fine-tuning changes the model's weights permanently through training. RAG never touches the model at all, it just feeds it better source material at the moment of the question. RAG is also a lot easier to update, adding a new article means adding it to the vector database, no retraining needed, where fine-tuning would need a whole new training run to include new information.

FAISS, ChromaDB, and Qdrant are three different vector databases with different tradeoffs. FAISS is a pure vector index library, extremely fast, but it doesn't store metadata alongside vectors on its own, you have to manage that separately. ChromaDB stores embeddings and their metadata together in one place and runs fully local with no separate server, which is why it got used here. Qdrant is a full production vector database server, built for handling much larger scale and more advanced filtering than a single local project needs.

## takeaway

30 articles turned into 282 chunks, embedded, stored, and searched, both queries pulling back their real source article by meaning instead of keywords. Day 5 builds on this same collection, an actual RAG chatbot that takes a question, retrieves chunks the same way these two queries did, and hands them to an LLM to answer from.